# Week 1 — Operations Analyzer

**Author:** _[Your Name]_

This notebook is a small toolkit for tracking how efficiently our sites are
running. If you're new to Python, read the Markdown cells like notes from
a teammate walking you through the logic before you look at the code.

**What "efficiency" means here:** how much a site actually produced,
compared to how much it was supposed to produce, shown as a percentage.
If a fuel depot was supposed to move 1,000 gallons today and it moved 850,
it ran at 85% efficiency.


## 1. KPI Function: `calculate_efficiency`

**What's a function, in plain terms?** It's a small, reusable recipe. You
give it some ingredients (here: `actual_output` and `target_output`), it
follows the same steps every time, and it hands back a result. We write it
once and reuse it for every site instead of retyping the same math
over and over.

**The math:** efficiency = (actual ÷ target) × 100.

**Edge case worth handling:** if `target_output` is 0, dividing by it would
crash the program (you can't divide by zero). We guard against that below
so one bad data row doesn't break the whole report.


In [ ]:
def calculate_efficiency(actual_output, target_output):
    """
    Calculate efficiency as a percentage of actual output vs. target output.

    Parameters
    ----------
    actual_output : float
        What the site actually produced/processed in the period.
    target_output : float
        What the site was expected/scheduled to produce in the period.

    Returns
    -------
    float
        Efficiency percentage, rounded to 1 decimal place.
        Returns 0.0 if target_output is 0, to avoid a divide-by-zero error.
    """
    if target_output == 0:
        # A target of 0 means "nothing was expected" -- efficiency is
        # undefined, so we report 0.0 rather than crashing the notebook.
        return 0.0

    efficiency = (actual_output / target_output) * 100
    return round(efficiency, 1)


# Quick sanity check -- always test a function right after you write it.
print(calculate_efficiency(850, 1000))   # expect 85.0
print(calculate_efficiency(0, 0))        # expect 0.0 (guarded, no crash)


## 2. Status Classifier: `get_operational_status`

**Why do we need this, if we already have a number?** A percentage like
"82.3%" doesn't tell a busy manager whether to worry. This function
translates the raw number into a plain-English label, using the
thresholds the business has agreed on:

- Below 70% → **Critical** (needs attention now)
- 70% up to 89% → **Warning** (keep an eye on it)
- 90% and above → **Normal** (running as expected)

**Note on the boundaries:** "between 70 and 89" is treated as inclusive on
both ends (70.0% counts as Warning, not Critical; 89.9% still counts as
Warning, not Normal), and 90.0% exactly counts as Normal. Getting boundary
conditions right matters -- this is exactly the kind of off-by-one detail
that causes real reporting bugs.


In [ ]:
def get_operational_status(efficiency):
    """
    Classify an efficiency percentage into a plain-English operational status.

    Parameters
    ----------
    efficiency : float
        Efficiency percentage, e.g. as returned by calculate_efficiency().

    Returns
    -------
    str
        One of "Critical", "Warning", or "Normal".
    """
    if efficiency < 70:
        return "Critical"
    elif efficiency < 90:          # covers the 70-89.9... range
        return "Warning"
    else:                          # 90 and above
        return "Normal"


# Quick sanity check across the boundaries
for test_value in [50, 69.9, 70, 89.9, 90, 100]:
    print(test_value, "->", get_operational_status(test_value))


## 3. Data Modeling: Our 3 Operational Sites

**Industry chosen: Fuel Depots.** We're tracking 3 depots and how much
fuel (in gallons) each moved today compared to its target.

**What's a dictionary, in plain terms?** Think of it as a labeled folder:
instead of remembering "the 3rd item in a list is the target," we give
each piece of information a name (a "key"), like `name`, `actual_output`,
and `target_output`, and look it up by that name. It reads almost like a
sentence: `site["name"]` literally means "give me this site's name."

Below, we have **one dictionary per site**, and all three sites are
collected into a list so we can loop over them in Section 4.


In [ ]:
depot_a = {
    "name": "Fuel Depot A - Riverside",
    "actual_output": 8500,   # gallons dispensed today
    "target_output": 10000,  # gallons scheduled for today
}

depot_b = {
    "name": "Fuel Depot B - Northgate",
    "actual_output": 9400,
    "target_output": 9500,
}

depot_c = {
    "name": "Fuel Depot C - Harbor",
    "actual_output": 5200,
    "target_output": 8000,
}

# Collect all sites into one list so Section 4 can loop through them.
sites = [depot_a, depot_b, depot_c]

sites


## 4. Execution: Generate the Report

**What's a loop, in plain terms?** It's an instruction to repeat the same
steps for every item in a group, one at a time, instead of copy-pasting
the same code 3 times. Here, "the same steps" are: calculate this site's
efficiency, classify its status, then print a line for it. The loop just
does that once per site, automatically.


In [ ]:
print("=" * 50)
print("DAILY OPERATIONAL EFFICIENCY REPORT")
print("=" * 50)

for site in sites:
    # Step 1: pull this site's numbers out of its dictionary
    name = site["name"]
    actual = site["actual_output"]
    target = site["target_output"]

    # Step 2: run our two functions from Sections 1 and 2
    efficiency = calculate_efficiency(actual, target)
    status = get_operational_status(efficiency)

    # Step 3: print a clean, formatted line for this site
    print(f"\nSite: {name}")
    print(f"  Actual Output:   {actual:,}")
    print(f"  Target Output:   {target:,}")
    print(f"  Efficiency:      {efficiency}%")
    print(f"  Status:          {status}")

print("\n" + "=" * 50)


**Reading the output:** each block above corresponds to one depot's
report card for the day. A junior analyst should be able to scan the
`Status` line alone and immediately know which depot(s) need a phone call
today -- that's the entire point of turning raw numbers into a status
label.


## 5. Recap (for the junior analyst reading this later)

1. `calculate_efficiency()` — a reusable recipe that turns two raw numbers
   (actual vs. target) into one percentage.
2. `get_operational_status()` — a reusable recipe that turns that
   percentage into a plain-English label using agreed thresholds.
3. Each site's data lives in its own **dictionary** (a labeled folder of
   `name` / `actual_output` / `target_output`), and all three sites sit
   together in a **list**.
4. The **loop** in Section 4 runs the same two functions on every site and
   prints a consistent report, so adding a 4th or 10th site later only
   means adding one more dictionary to the `sites` list -- no other code
   needs to change.

This is the core pattern behind almost every automated reporting tool:
**small reusable functions + structured data + a loop that ties them
together.**
